In [1]:
# -*- coding: utf-8 -*-
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ─── IMPORTS ──────────────────────────────────────────────────────────────────
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mnv3_prep
from tensorflow.keras.applications.convnext import preprocess_input as convnext_prep
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as effv2_prep
from tensorflow.keras.applications.densenet import preprocess_input as densenet_prep

In [3]:
# ─── CONFIG ───────────────────────────────────────────────────────────────────
TEST_DIR = "/content/drive/MyDrive/Dataset/CLASSIFICATION_Epic and CSCR hospital Dataset_clean/Test"

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]

MODEL_PATHS = {
    "MobileNetV3"     : "/content/drive/MyDrive/Project work/models/class_Tumor_mobilenet_v3.keras",
    "ConvNeXt-Tiny"   : "/content/drive/MyDrive/Project work/models/class_Tumor_convnext_tiny_tumor.keras",
    "EfficientNetV2-S": "/content/drive/MyDrive/Project work/models/class_Tumor_v2s_clean.keras",
    "DenseNet201"     : "/content/drive/MyDrive/Project work/models/class_Tumor_densenet_201.keras",
}

PREPROCESS = {
    "MobileNetV3"     : mnv3_prep,
    "ConvNeXt-Tiny"   : convnext_prep,
    "EfficientNetV2-S": effv2_prep,
    "DenseNet201"     : densenet_prep,
}

IMG_SIZES = {
    "MobileNetV3"     : 384,
    "ConvNeXt-Tiny"   : 384,
    "EfficientNetV2-S": 384,
    "DenseNet201"     : 224,
}

BATCH_SIZE = 32

In [ ]:
# ─── EVALUATE ─────────────────────────────────────────────────────────────────
results = []

for model_name, model_path in MODEL_PATHS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")

    img_size = IMG_SIZES[model_name]
    preprocess_fn = PREPROCESS[model_name]

    # Build generator for this model's image size
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    gen = ImageDataGenerator(preprocessing_function=preprocess_fn)

    test_gen = gen.flow_from_directory(
        TEST_DIR,
        target_size=(img_size, img_size),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=False,
        classes=CLASS_NAMES
    )

    true_labels = test_gen.classes

    # Load and predict
    model = tf.keras.models.load_model(model_path, compile=False)
    test_gen.reset()
    preds = model.predict(test_gen, verbose=1)
    pred_labels = np.argmax(preds, axis=1)

    # Metrics
    acc  = accuracy_score(true_labels, pred_labels)
    prec = precision_score(true_labels, pred_labels, average="weighted", zero_division=0)
    rec  = recall_score(true_labels, pred_labels, average="weighted", zero_division=0)
    f1   = f1_score(true_labels, pred_labels, average="weighted", zero_division=0)

    print(f"\nAccuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"\nPer-Class Report:")
    print(classification_report(true_labels, pred_labels, target_names=CLASS_NAMES, zero_division=0))

    results.append({
        "Model"    : model_name,
        "Accuracy" : round(acc * 100, 2),
        "Precision": round(prec * 100, 2),
        "Recall"   : round(rec * 100, 2),
        "F1-Score" : round(f1 * 100, 2)
    })

    del model  # free GPU memory


Evaluating: MobileNetV3
Found 2180 images belonging to 4 classes.
69/69 ━━━━━━━━━━━━━━━━━━━━ 674s 10s/step

Accuracy  : 0.9206
Precision : 0.9212
Recall    : 0.9206
F1-Score  : 0.9198

Per-Class Report:
              precision    recall  f1-score   support

      glioma       0.95      0.87      0.91       750
  meningioma       0.88      0.87      0.88       518
     notumor       0.93      0.99      0.96       455
   pituitary       0.92      1.00      0.95       457

    accuracy                           0.92      2180
   macro avg       0.92      0.93      0.92      2180
weighted avg       0.92      0.92      0.92      2180


Evaluating: ConvNeXt-Tiny
Found 2180 images belonging to 4 classes.
69/69 ━━━━━━━━━━━━━━━━━━━━ 11456s 166s/step

Accuracy  : 0.9394
Precision : 0.9408
Recall    : 0.9394
F1-Score  : 0.9391

Per-Class Report:
              precision    recall  f1-score   support

      glioma       0.95      0.91      0.93       750
  meningioma       0.95      0.89      0.92

In [ ]:
# ─── SUMMARY TABLE ────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL SUMMARY TABLE (for paper)")
print("="*60)
df = pd.DataFrame(results)
print(df.to_string(index=False))
print("\nNote: All values are in percentage (%)")


FINAL SUMMARY TABLE (for paper)
           Model  Accuracy  Precision  Recall  F1-Score
     MobileNetV3     92.06      92.12   92.06     91.98
   ConvNeXt-Tiny     93.94      94.08   93.94     93.91
EfficientNetV2-S     97.39      97.39   97.39     97.38
     DenseNet201     44.27      60.82   44.27     40.16

Note: All values are in percentage (%)
